# `utils.py` — Pipeline unificado de datos y métricas

Archivo centralizado para el proyecto de CRS. Contiene:
1. Utilidades generales
2. Descarga y carga de datasets (ReDIAL, PEARL)
3. Construcción de contexto y catálogos
4. Artefactos de datos para métricas
5. Métricas con soft-matching (Recall@K, MRR, Novelty, BERTScore)

# Guía de uso de `utils.ipynb`

## Setup inicial (en cada notebook)

Antes de usar cualquier función, descargar utils.ipynb al notebook respectivo y agregar al inicio:

```python
from utils import (
    download_dataset, load_parsed, sample_conversations,
    build_context, build_global_catalog,
    build_train_freq, build_recommender_references,
    evaluate_soft, evaluate_novelty, evaluate_bertscore, run_full_evaluation,
)
```
---

## Métricas: evaluación

### Requisitos previos

Las métricas de soft-matching necesitan un modelo de embeddings. **Este modelo se carga una sola vez en cada notebook** (no vive dentro de utils para no disparar una carga pesada al importar):

```python
from sentence_transformers import SentenceTransformer
import torch

device_embed = "cuda" if torch.cuda.is_available() else "cpu"
EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2", device=device_embed)
```

Para Novelty y BERTScore, se necesitan artefactos de datos que también se preparan una sola vez:

```python
train_freq, n_train = build_train_freq(paths["train"])
human_refs = build_recommender_references(paths["test"])
```

### Formato esperado de `outputs`

Todas las funciones de métricas esperan una lista de dicts con esta estructura:

```python
{
    "idx": int,             # índice en el parsed original
    "predicted": list[str], # títulos predichos, ej. ["The Matrix (1999)", "Inception (2010)", ...]
    "message": str,         # mensaje generado por el modelo (usado por BERTScore)
}
```
---

## Parámetros importantes

| Parámetro | Valor | Notas |
|---|---|---|
| `threshold` | `0.90` | Umbral de similitud coseno para soft-matching. Fijado para todos los experimentos. |
| `gt_field` | `"gt_accepted"` | Ground truth estricto: sugerido por recommender Y aceptado por seeker. |
| `embed_model` | `all-MiniLM-L6-v2` | Modelo para soft-matching de Recall y MRR. |
| `model_type` (BERTScore) | `roberta-base` | Modelo para BERTScore. Usar el mismo para todos los métodos. |

**Regla clave:** estos valores deben ser idénticos en todos los notebooks para que los resultados sean comparables.

## 1. Imports y utilidades generales

In [1]:
import urllib.request
import json
import re
import os
import math
import random
import numpy as np
import torch


def _norm_title(title: str) -> str:
    """Normaliza un título de película.
    'The Matrix (1999)' → 'the matrix'
    """
    return title.split(" (")[0].strip().lower()

## 2. Descarga y carga de datasets

In [2]:
# ── URLs conocidas de los datasets ──────────────────────────────────
DATASET_URLS = {
    "redial": "https://huggingface.co/datasets/recwizard/redial/resolve/main/",
    "pearl": "https://huggingface.co/datasets/LangAGI-Lab/pearl/resolve/main/",
}

# Extensión de archivo por dataset (PEARL usa .json, ReDIAL usa .jsonl)
_DATASET_EXT = {
    "redial": "jsonl",
    "pearl": "json",
}


def download_dataset(dataset: str = "redial", splits: tuple = ("train", "test")) -> dict:
    """Descarga los splits de un dataset desde HuggingFace.

    Args:
        dataset: Nombre del dataset ("redial" o "pearl").
        splits: Tupla con los splits a descargar.

    Returns:
        Dict {split: path_local} con las rutas a los archivos descargados.
    """
    if dataset not in DATASET_URLS:
        raise ValueError(f"Dataset '{dataset}' no soportado. Opciones: {list(DATASET_URLS.keys())}")

    base = DATASET_URLS[dataset]
    ext = _DATASET_EXT.get(dataset, "jsonl")
    paths = {}
    for split in splits:
        filename = f"{dataset}_{split}.{ext}"
        if not os.path.exists(filename):
            print(f"Descargando {filename}...")
            urllib.request.urlretrieve(base + f"{split}.{ext}", filename)
        else:
            print(f"{filename} ya existe, omitiendo descarga.")
        paths[split] = filename
    return paths

In [3]:
def load_parsed(path: str, dataset: str = "redial") -> list[dict]:
    """Parsea un archivo de ReDIAL (.jsonl) o PEARL (.json) y extrae la información relevante.

    Cada diálogo se convierte en un dict con:
        - n_turns: número total de turnos
        - seeker_movies: IDs de películas mencionadas por el seeker (no sugeridas)
        - gt_suggested: IDs sugeridas por el recommender (excluyendo las del seeker)
        - gt_seeker_liked: IDs que el seeker marcó como liked
        - gt_accepted: intersección de gt_suggested y gt_seeker_liked (GT más estricto)
        - title_map: {movie_id: "Title (Year)"}
        - messages: lista de {"role": "seeker"|"recommender", "text": str}

    Args:
        path: Ruta al archivo .jsonl (ReDIAL) o .json (PEARL).
        dataset: "redial" o "pearl". Default "redial" para compatibilidad.

    Returns:
        Lista de dicts, uno por diálogo.
    """
    if dataset == "pearl":
        return _load_parsed_pearl(path)
    return _load_parsed_redial(path)


def _load_parsed_redial(path: str) -> list[dict]:
    """Parser interno para ReDIAL (.jsonl)."""
    parsed = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            title_map = {str(k): v for k, v in (d.get("movieMentions") or {}).items()}
            resp_ann = d.get("respondentQuestions") or {}
            init_ann = d.get("initiatorQuestions") or {}
            initiator_id = d.get("initiatorWorkerId")

            def norm(mid):
                return title_map.get(mid, "").split(" (")[0].strip().lower()

            def replace_ids(text):
                return re.sub(r'@(\d+)', lambda m: title_map.get(m.group(1), ''), text)

            # ── Películas del seeker (no sugeridas) ──
            seeker_movies = [mid for mid, a in resp_ann.items() if a.get("suggested") == 0]
            seeker_set = set(seeker_movies)
            seeker_titles = {norm(mid) for mid in seeker_movies}
            seeker_titles.discard("")

            # ── Ground truths ──
            gt_suggested = [mid for mid, a in resp_ann.items()
                            if a.get("suggested") == 1
                            and mid not in seeker_set
                            and norm(mid) not in seeker_titles]

            gt_seeker_liked = [mid for mid, a in init_ann.items()
                               if a.get("liked") == 1
                               and mid not in seeker_set
                               and norm(mid) not in seeker_titles]

            gt_suggested_set = set(gt_suggested)
            gt_accepted = [mid for mid in gt_seeker_liked if mid in gt_suggested_set]

            # ── Historial conversacional completo con roles ──
            messages_clean = []
            for msg in d.get("messages", []):
                text = replace_ids(msg.get("text", "")).strip()
                if text:
                    role = "seeker" if msg.get("senderWorkerId") == initiator_id else "recommender"
                    messages_clean.append({"role": role, "text": text})

            parsed.append({
                "n_turns": len(d.get("messages", [])),
                "seeker_movies": seeker_movies,
                "gt_suggested": gt_suggested,
                "gt_seeker_liked": gt_seeker_liked,
                "gt_accepted": gt_accepted,
                "title_map": title_map,
                "messages": messages_clean,
            })
    return parsed


def _load_parsed_pearl(path: str) -> list[dict]:
    """Parser interno para PEARL (.json).

    PEARL tiene un solo gt_movie_title por diálogo.
    Se asume que es sugerido, liked y aceptado (las tres condiciones).
    """
    parsed = []
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    for d in data:
        seen = d.get("seen_movie_titles", [])
        gt_title = d.get("gt_movie_title", "")

        # ── IDs sintéticos (PEARL no tiene IDs numéricos) ──
        title_map = {}
        seeker_movies = []
        for j, t in enumerate(seen):
            mid = f"s{j}"
            title_map[mid] = t
            seeker_movies.append(mid)

        gt_id = "gt0"
        if gt_title:
            title_map[gt_id] = gt_title

        # ── Ground truths ──
        # PEARL: gt_movie_title es la recomendación correcta del diálogo.
        # Se trata como sugerida por el recommender Y aceptada por el seeker.
        gt_suggested = [gt_id] if gt_title else []
        gt_seeker_liked = [gt_id] if gt_title else []   # asumido: seeker aceptó la recomendación
        gt_accepted = [gt_id] if gt_title else []        # asumido: intersección = mismo gt

        # ── Historial conversacional ──
        messages_clean = []
        for msg_str in d.get("dialogue", []):
            if msg_str.startswith("Seeker: "):
                role = "seeker"
                text = msg_str[len("Seeker: "):].strip()
            elif msg_str.startswith("Recommender: "):
                role = "recommender"
                text = msg_str[len("Recommender: "):].strip()
            else:
                continue
            if text:
                messages_clean.append({"role": role, "text": text})

        parsed.append({
            "n_turns": len(d.get("dialogue", [])),
            "seeker_movies": seeker_movies,
            "gt_suggested": gt_suggested,
            "gt_seeker_liked": gt_seeker_liked,
            "gt_accepted": gt_accepted,
            "title_map": title_map,
            "messages": messages_clean,
        })
    return parsed

In [4]:
def sample_conversations(parsed: list, n: int, seed: int = 42,
                         gt_field: str = "gt_accepted") -> list[dict]:
    """Samplea n conversaciones aleatorias que tengan ground truth no vacío.

    Cada conversación retornada incluye el campo 'original_idx' con su
    índice en la lista parsed original, para trazabilidad.

    Args:
        parsed: Lista completa de diálogos parseados.
        n: Cantidad de conversaciones a samplear.
        seed: Semilla para reproducibilidad.
        gt_field: Campo de ground truth que debe ser no vacío.

    Returns:
        Lista de n dicts (subconjunto de parsed) con campo extra 'original_idx'.
    """
    with_gt = [(i, d) for i, d in enumerate(parsed) if d.get(gt_field)]
    if len(with_gt) < n:
        print(f"Advertencia: solo hay {len(with_gt)} conversaciones con {gt_field} no vacío, "
              f"se pidieron {n}. Se retornan todas.")

    rng = random.Random(seed)
    selected = rng.sample(with_gt, min(n, len(with_gt)))

    result = []
    for orig_idx, d in selected:
        entry = dict(d)
        entry["original_idx"] = orig_idx
        result.append(entry)

    print(f"Sampled {len(result)} conversaciones con {gt_field} no vacío (seed={seed})")
    return result

## 3. Construcción de contexto y catálogos

In [5]:
def build_context(dialogue: dict) -> str:
    """Construye el contexto conversacional para inyectar en el prompt.

    Usa la conversación completa EXCEPTO:
    - El último mensaje del recommender (que contiene la recomendación a predecir).
    - Cualquier mensaje del seeker que venga después de ese último turno del recommender.

    Es decir, trunca la conversación justo antes del último turno del recommender.

    Args:
        dialogue: Dict de un diálogo parseado (con campo 'messages').

    Returns:
        String formateado con el historial conversacional.
    """
    messages = dialogue.get("messages", [])

    if not messages:
        return _context_fallback(dialogue)

    # Encontrar el índice del último mensaje del recommender
    last_rec_idx = None
    for i in range(len(messages) - 1, -1, -1):
        if messages[i]["role"] == "recommender":
            last_rec_idx = i
            break

    # Truncar desde el último mensaje del recommender en adelante
    if last_rec_idx is not None:
        messages = messages[:last_rec_idx]

    if not messages:
        return _context_fallback(dialogue)

    # return "Conversation history:\n" + "\n".join(
    #     f"- {m['text']}" for m in messages
    # )

    # darle user y assistant para que el LLM entienda que es conversacion
    return "Conversation history:\n" + "\n".join(
        f"{'User' if m['role'] == 'seeker' else 'Assistant'}: {m['text']}"
        for m in messages
    )


def _context_fallback(dialogue: dict) -> str:
    """Fallback cuando no hay mensajes útiles: usa las películas del seeker."""
    titles = [dialogue["title_map"].get(mid, "") for mid in dialogue.get("seeker_movies", [])]
    titles = [t for t in titles if t]
    if titles:
        return "The user has mentioned liking these movies: " + ", ".join(titles) + "."
    return "The user is looking for a movie recommendation."

In [6]:
def build_global_catalog(*parsed_lists) -> dict:
    """Construye un catálogo unificado de películas a partir de múltiples parsed lists.

    Args:
        *parsed_lists: Una o más listas de diálogos parseados (ej. train_parsed, test_parsed).

    Returns:
        Dict {movie_id: "Title (Year)"} con todas las películas encontradas.
    """
    catalog = {}
    for parsed in parsed_lists:
        for d in parsed:
            catalog.update(d["title_map"])
    print(f"Catálogo global: {len(catalog)} películas")
    return catalog

## 4. Artefactos de datos para métricas

In [7]:
def build_train_freq(train_path: str, dataset: str = "redial") -> tuple[dict, int]:
    """Calcula la document-frequency de cada título en el set de entrenamiento.

    Cuenta cuántos diálogos mencionan cada película (1 por diálogo máximo).
    Necesaria para calcular Novelty.

    Args:
        train_path: Ruta al archivo de entrenamiento.
        dataset: "redial" o "pearl".

    Returns:
        Tupla (freq_dict, n_dialogos):
            - freq_dict: {título_normalizado: n_diálogos_que_lo_mencionan}
            - n_dialogos: total de diálogos en el train set
    """
    freq, n = {}, 0
    with open(train_path, encoding="utf-8") as f:
        if dataset == "pearl":
            data = json.load(f)
            for d in data:
                n += 1
                titles = set()
                for t in d.get("seen_movie_titles", []):
                    nt = _norm_title(t)
                    if nt:
                        titles.add(nt)
                gt = _norm_title(d.get("gt_movie_title", ""))
                if gt:
                    titles.add(gt)
                for t in titles:
                    freq[t] = freq.get(t, 0) + 1
        else:  # redial
            for line in f:
                d = json.loads(line)
                n += 1
                tmap = {str(k): v for k, v in (d.get("movieMentions") or {}).items()}
                titles = {_norm_title(v) for v in tmap.values() if v}
                titles.discard("")
                for t in titles:
                    freq[t] = freq.get(t, 0) + 1
    print(f"Train freq construido: {len(freq)} títulos únicos en {n} diálogos")
    return freq, n

In [8]:
def build_recommender_references(path_jsonl: str, only_movie_turns: bool = True,
                                  dataset: str = "redial") -> list[str]:
    """Extrae las respuestas del recomendador humano como referencias para BERTScore.

    Args:
        path_jsonl: Ruta al archivo (típicamente test).
        only_movie_turns: Si True, solo incluye turnos que mencionan una película.
        dataset: "redial" o "pearl".

    Returns:
        Lista de strings alineada por índice de diálogo. Cada string es la
        concatenación de los turnos relevantes del recomendador humano.
    """
    refs = []
    with open(path_jsonl, encoding="utf-8") as f:
        if dataset == "pearl":
            data = json.load(f)
            for d in data:
                # Títulos conocidos para filtrado de movie_turns
                known_titles = set()
                for t in d.get("seen_movie_titles", []):
                    nt = _norm_title(t)
                    if nt:
                        known_titles.add(nt)
                gt = _norm_title(d.get("gt_movie_title", ""))
                if gt:
                    known_titles.add(gt)

                turns = []
                for msg_str in d.get("dialogue", []):
                    if not msg_str.startswith("Recommender: "):
                        continue
                    text = msg_str[len("Recommender: "):].strip()
                    if only_movie_turns:
                        text_lower = text.lower()
                        if not any(t in text_lower for t in known_titles):
                            continue
                    if text:
                        turns.append(text)
                refs.append(" ".join(turns).strip())
        else:  # redial
            for line in f:
                d = json.loads(line)
                tmap = {str(k): v for k, v in (d.get("movieMentions") or {}).items()}
                resp_id = d.get("respondentWorkerId")

                def repl(text):
                    return re.sub(r'@(\d+)', lambda m: tmap.get(m.group(1), ''), text)

                turns = []
                for m in d.get("messages", []):
                    if m.get("senderWorkerId") != resp_id:
                        continue
                    raw = m.get("text", "")
                    if only_movie_turns and "@" not in raw:
                        continue
                    t = repl(raw).strip()
                    if t:
                        turns.append(t)
                refs.append(" ".join(turns).strip())
    print(f"Referencias construidas: {len(refs)} diálogos")
    return refs

## 5. Métricas con soft-matching

In [9]:
def evaluate_soft(outputs: list, parsed: list, embed_model,
                  threshold: float = 0.90, gt_field: str = "gt_accepted",
                  k_list: tuple = (1, 5)) -> dict:
    """Calcula Recall@K y MRR con soft-matching semántico en una sola pasada.

    Usa similitud coseno entre embeddings de predicciones y ground truth.
    Un hit se cuenta cuando la similitud >= threshold.

    Args:
        outputs: Lista de dicts {"idx": int, "predicted": list[str], ...}.
        parsed: Dataset parseado completo (para acceder a title_map y GT).
        embed_model: Instancia de SentenceTransformer (se carga externamente).
        threshold: Umbral de similitud coseno para considerar un match.
        gt_field: Campo del ground truth a evaluar.
        k_list: Valores de K para Recall@K.

    Returns:
        Dict con Recall@1, Recall@5, MRR y n_evaluable.
    """
    results = {f"Recall@{k}": [] for k in k_list}
    results["MRR"] = []
    max_k = max(k_list)

    for o in outputs:
        d = parsed[o["idx"]]
        gt_titles = [d["title_map"].get(mid, "").split(" (")[0].strip()
                     for mid in d[gt_field]]
        gt_titles = [t for t in gt_titles if t]

        if not gt_titles:
            continue

        pred = [p.split(" (")[0].strip() for p in o["predicted"][:max_k]]

        if not pred:
            for k in k_list:
                results[f"Recall@{k}"].append(0.0)
            results["MRR"].append(0.0)
            continue

        # Embeddings + normalización
        pred_embs = embed_model.encode(pred, convert_to_tensor=True, show_progress_bar=False)
        gt_embs = embed_model.encode(gt_titles, convert_to_tensor=True, show_progress_bar=False)

        pred_embs = torch.nn.functional.normalize(pred_embs, p=2, dim=1)
        gt_embs = torch.nn.functional.normalize(gt_embs, p=2, dim=1)

        # Similitud: para cada predicción, máxima similitud contra cualquier GT
        sim_matrix = torch.mm(pred_embs, gt_embs.transpose(0, 1))
        max_sims = sim_matrix.max(dim=1).values
        hits = (max_sims >= threshold).tolist()

        # Recall@K: ¿hay al menos un hit en los top-K?
        for k in k_list:
            results[f"Recall@{k}"].append(1.0 if any(hits[:k]) else 0.0)

        # MRR: posición del primer hit
        first_hit = next((i + 1 for i, h in enumerate(hits) if h), None)
        results["MRR"].append(1.0 / first_hit if first_hit else 0.0)

    n_evaluable = len(results["MRR"])
    final = {m: round(float(np.mean(v)), 4) for m, v in results.items()}
    final["n_evaluable"] = n_evaluable

    print(f"Evaluable: {n_evaluable} / {len(outputs)}")
    for k in k_list:
        print(f"  Recall@{k}: {final[f'Recall@{k}']:.4f}")
    print(f"  MRR:      {final['MRR']:.4f}")

    return final

In [10]:
def evaluate_novelty(outputs: list, train_freq: dict, n_train: int,
                     k: int = 5) -> dict:
    """Calcula la novelty (auto-información) de las recomendaciones.

    Mayor novelty = película menos popular en train = recomendación más novedosa.
    También reporta la tasa de películas no vistas en train (posibles alucinaciones).

    Args:
        outputs: Lista de dicts con campo "predicted".
        train_freq: Dict de frecuencias del train set (de build_train_freq).
        n_train: Número total de diálogos en train.
        k: Cuántas predicciones considerar por diálogo.

    Returns:
        Dict con novelty_mean, novelty_norm, unseen_rate, n_dialogos, n_recs.
    """
    max_nov = -math.log2(1 / (n_train + 1))
    per_dialogue = []
    n_recs = n_unseen = n_dialogos = 0

    for o in outputs:
        preds = o["predicted"][:k]
        if not preds:
            continue
        n_dialogos += 1
        novs = []
        for p in preds:
            t = _norm_title(p)
            df = train_freq.get(t, 0)
            if df == 0:
                n_unseen += 1
            prob = (df + 1) / (n_train + 1)  # suavizado Laplace
            novs.append(-math.log2(prob))
            n_recs += 1
        per_dialogue.append(float(np.mean(novs)))

    mean_nov = float(np.mean(per_dialogue)) if per_dialogue else 0.0
    unseen_rate = n_unseen / n_recs if n_recs else 0.0

    result = {
        "novelty_mean": round(mean_nov, 4),
        "novelty_norm": round(mean_nov / max_nov, 4) if max_nov else 0.0,
        "unseen_rate": round(unseen_rate, 4),
        "n_dialogos": n_dialogos,
        "n_recs": n_recs,
    }

    print(f"  Novelty:  {result['novelty_mean']:.2f} bits (norm: {result['novelty_norm']:.3f})")
    print(f"  No vistas en train: {unseen_rate * 100:.1f}% ({n_unseen}/{n_recs})")

    return result

In [11]:
def evaluate_bertscore(outputs: list, references: list, lang: str = "en",
                       model_type: str = "roberta-base",
                       rescale_with_baseline: bool = False) -> dict:
    """Calcula BERTScore entre el mensaje del modelo y la referencia humana.

    Args:
        outputs: Lista de dicts con campo "message".
        references: Lista de strings alineada por idx (de build_recommender_references).
        lang: Idioma para BERTScore.
        model_type: Modelo a usar para los embeddings de BERTScore.
        rescale_with_baseline: Si True, reescala los scores con baselines precomputados.

    Returns:
        Dict con precision, recall, f1, n_evaluable, n_total.
    """
    import evaluate as hf_evaluate

    bertscore = hf_evaluate.load("bertscore")
    preds, refs, used = [], [], 0

    for o in outputs:
        pred = (o.get("message") or "").strip()
        ref = references[o["idx"]].strip()
        if pred and ref:
            preds.append(pred)
            refs.append(ref)
            used += 1

    if used == 0:
        print("BERTScore: sin pares evaluables.")
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0,
                "n_evaluable": 0, "n_total": len(outputs)}

    res = bertscore.compute(
        predictions=preds, references=refs, lang=lang,
        model_type=model_type, rescale_with_baseline=rescale_with_baseline
    )

    result = {
        "precision": round(float(np.mean(res["precision"])), 4),
        "recall":    round(float(np.mean(res["recall"])), 4),
        "f1":        round(float(np.mean(res["f1"])), 4),
        "n_evaluable": used,
        "n_total": len(outputs),
    }

    print(f"  BERTScore ({used}/{len(outputs)}): "
          f"P={result['precision']:.4f}  R={result['recall']:.4f}  F1={result['f1']:.4f}")

    return result

In [12]:
def run_full_evaluation(outputs: list, parsed: list, embed_model,
                        train_freq: dict, n_train: int, references: list,
                        threshold: float = 0.90, gt_field: str = "gt_accepted",
                        k_list: tuple = (1, 5)) -> dict:
    """Ejecuta todas las métricas en una sola llamada y retorna un dict consolidado.

    Args:
        outputs: Lista de dicts {"idx", "predicted", "message", ...}.
        parsed: Dataset parseado completo.
        embed_model: Instancia de SentenceTransformer.
        train_freq: Frecuencias del train (de build_train_freq).
        n_train: N diálogos en train.
        references: Referencias humanas (de build_recommender_references).
        threshold: Umbral de similitud para soft-matching.
        gt_field: Campo del GT.
        k_list: Valores de K para Recall@K.

    Returns:
        Dict con todas las métricas consolidadas.
    """
    print("=" * 55)
    print("  EVALUACIÓN COMPLETA")
    print("=" * 55)

    print("\n── Recall & MRR (soft-matching) ──")
    soft = evaluate_soft(outputs, parsed, embed_model,
                         threshold=threshold, gt_field=gt_field, k_list=k_list)

    print("\n── Novelty ──")
    nov = evaluate_novelty(outputs, train_freq, n_train)

    print("\n── BERTScore ──")
    bert = evaluate_bertscore(outputs, references)

    print("\n" + "=" * 55)

    return {**soft, **nov, **bert}

In [13]:
def soft_hit(pred_title: str, gt_titles: list[str], embed_model,
             threshold: float = 0.90) -> bool:   # para match de tittles en ACE
    """Determina si pred_title matchea (soft-match) alguno de gt_titles.

    Usa el MISMO criterio que evaluate_soft (cosine similarity sobre
    embeddings normalizados de all-MiniLM-L6-v2), para que el hit-signal
    usado durante warmup (ACE, Reflexion, DC) sea consistente con la
    métrica final reportada.

    Args:
        pred_title: título predicho (con o sin año, se normaliza internamente).
        gt_titles: lista de títulos ground truth a comparar.
        embed_model: instancia de SentenceTransformer ya cargada.
        threshold: umbral de similitud coseno (default 0.90, igual que evaluate_soft).

    Returns:
        True si la similitud máxima contra cualquier gt_title >= threshold.
    """
    if not pred_title or not gt_titles:
        return False

    p = _norm_title(pred_title)
    gts = [_norm_title(g) for g in gt_titles]
    gts = [g for g in gts if g]
    if not p or not gts:
        return False

    if p in gts:           # match exacto -> ahorra encode innecesario
        return True

    pred_emb = embed_model.encode([p], convert_to_tensor=True, show_progress_bar=False)
    gt_embs  = embed_model.encode(gts, convert_to_tensor=True, show_progress_bar=False)

    pred_emb = torch.nn.functional.normalize(pred_emb, p=2, dim=1)
    gt_embs  = torch.nn.functional.normalize(gt_embs, p=2, dim=1)

    sim = torch.mm(pred_emb, gt_embs.transpose(0, 1)).max().item()
    return sim >= threshold

---
# **COMENTAR ESTA SECCIÓN ANTES DE DESCARGAR, SINO TIRA ERROR AL IMPORTARLO** ⬇
---

## 6. Test rápido carga de datos

Celdas para verificar que todo funciona correctamente con una muestra pequeña.

In [14]:
# # ── Test: descarga, carga y sampling ──
# if __name__ == "__main__":
#     paths = download_dataset("redial", splits=("train", "test"))

#     train_parsed = load_parsed(paths["train"])
#     test_parsed = load_parsed(paths["test"])

#     print(f"Train: {len(train_parsed)} | Test: {len(test_parsed)}")
#     print(f"Test con gt_accepted: {sum(1 for d in test_parsed if d['gt_accepted'])}")

#     # Verificar estructura de messages
#     sample_msg = test_parsed[0]["messages"][0]
#     print(f"Estructura de mensaje: {sample_msg}")
#     assert "role" in sample_msg and "text" in sample_msg, "Formato de mensajes incorrecto"

#     # Sampling
#     eval_sample = sample_conversations(test_parsed, n=300, gt_field="gt_accepted")
#     warmup_sample = sample_conversations(train_parsed, n=100, gt_field="gt_accepted")
#     print(f"Eval sample: {len(eval_sample)} | Warmup sample: {len(warmup_sample)}")

#     # Contexto
#     ctx = build_context(test_parsed[0])
#     print(f"\nContexto (primeros 300 chars):\n{ctx[:300]}...")

#     # Catálogo
#     catalog = build_global_catalog(train_parsed, test_parsed)

## 7. Test rápido Métricas

Celdas para verificar que todo funciona correctamente con una muestra falsa.

In [15]:
# # ── 7a. Preparar outputs random para test de métricas ──────────────
# from sentence_transformers import SentenceTransformer

# # Modelo de embeddings (no LLM, es liviano)
# device_embed = "cuda" if torch.cuda.is_available() else "cpu"
# EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2", device=device_embed)

# # Artefactos de datos
# train_freq, n_train = build_train_freq(paths["train"])
# human_refs = build_recommender_references(paths["test"])

# # Catálogo como lista para samplear
# catalog_titles = [v for v in catalog.values() if v and v.strip()]

# # Generar outputs random con la misma estructura que producen los métodos reales
# N_TEST = 30  # suficiente para verificar que las métricas corren
# random.seed(42)
# fake_outputs = []
# for i, d in enumerate(test_parsed[:N_TEST]):
#     picks = random.sample(catalog_titles, min(5, len(catalog_titles)))
#     fake_outputs.append({
#         "idx": i,
#         "predicted": picks,
#         "message": f"I recommend {picks[0]}. You might also enjoy {', '.join(picks[1:])}.",
#     })

# print(f"Outputs generados: {len(fake_outputs)}")
# print(f"Ejemplo: {fake_outputs[0]}")

In [16]:
# # ── 7b. Correr todas las métricas ──────────────────────────────────
# !pip install -q evaluate bert_score

# # Individualmente primero
# print("── evaluate_soft ──")
# soft = evaluate_soft(fake_outputs, test_parsed, EMBED_MODEL,
#                      threshold=0.90, gt_field="gt_accepted")

# print("\n── evaluate_novelty ──")
# nov = evaluate_novelty(fake_outputs, train_freq, n_train)

# print("\n── evaluate_bertscore ──")
# bert = evaluate_bertscore(fake_outputs, human_refs)

# # Consolidada
# print("\n\n── run_full_evaluation ──")
# all_metrics = run_full_evaluation(
#     fake_outputs, test_parsed, EMBED_MODEL,
#     train_freq, n_train, human_refs,
#     threshold=0.90, gt_field="gt_accepted"
# )

# # Sanity check
# print("\n── Sanity check ──")
# assert soft["Recall@1"] < 0.15, f"Recall@1 sospechosamente alto para random: {soft['Recall@1']}"
# assert soft["Recall@5"] < 0.30, f"Recall@5 sospechosamente alto para random: {soft['Recall@5']}"
# assert nov["novelty_norm"] > 0.5, f"Novelty sospechosamente baja para random: {nov['novelty_norm']}"
# print("✅ Todas las métricas corrieron correctamente y los valores son coherentes con random.")

# EXTRA: Justificación umbral 0.9 en soft-match

In [17]:
# # ── Análisis empírico del umbral de similitud para soft-matching ──────────
# # Objetivo: inspeccionar pares (predicción, GT) en distintos rangos de similitud
# # para justificar la elección del threshold en el informe.

# from sentence_transformers import SentenceTransformer, util as st_util
# import random, numpy as np

# device_embed = "cuda" if torch.cuda.is_available() else "cpu"
# embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=device_embed)

# # Recolectar pares (pred_title, gt_title, similitud) usando outputs random
# random.seed(123)
# all_pairs = []

# sample_indices = random.sample(range(len(test_parsed)), min(300, len(test_parsed)))
# for i in sample_indices:
#     d = test_parsed[i]
#     gt_titles = [d["title_map"].get(mid, "").split(" (")[0].strip()
#                  for mid in d["gt_accepted"]]
#     gt_titles = [t for t in gt_titles if t]
#     if not gt_titles:
#         continue

#     # Predicciones: mezcla de películas random del catálogo + algunas del GT con ruido
#     # para generar pares en todos los rangos de similitud
#     catalog_sample = random.sample(catalog_titles, min(5, len(catalog_titles)))
#     pred_titles = [t.split(" (")[0].strip() for t in catalog_sample]

#     # Agregar variantes del GT para tener pares de alta similitud
#     for gt in gt_titles[:2]:
#         pred_titles.append(gt)                          # exacto → ~1.0
#         pred_titles.append(gt + " 2")                   # secuela falsa → ~0.85-0.95
#         pred_titles.append("The " + gt)                 # prefijo → ~0.90-0.98

#     pred_embs = embed_model.encode(pred_titles, convert_to_tensor=True, show_progress_bar=False)
#     gt_embs = embed_model.encode(gt_titles, convert_to_tensor=True, show_progress_bar=False)
#     sims = st_util.cos_sim(pred_embs, gt_embs)

#     for pi in range(len(pred_titles)):
#         for gi in range(len(gt_titles)):
#             sim = sims[pi][gi].item()
#             all_pairs.append((pred_titles[pi], gt_titles[gi], sim))

# # Clasificar por rangos
# rangos = [
#     ("0.95 - 1.00", 0.95, 1.01),
#     ("0.90 - 0.95", 0.90, 0.95),
#     ("0.85 - 0.90", 0.85, 0.90),
#     ("0.80 - 0.85", 0.80, 0.85),
#     ("0.75 - 0.80", 0.75, 0.80),
# ]

# print("=" * 80)
# print("  ANÁLISIS EMPÍRICO DEL UMBRAL DE SIMILITUD (all-MiniLM-L6-v2)")
# print("=" * 80)

# for label, low, high in rangos:
#     en_rango = [(p, g, s) for p, g, s in all_pairs if low <= s < high]
#     print(f"\n{'─' * 80}")
#     print(f"  Rango {label}  |  {len(en_rango)} pares encontrados")
#     print(f"{'─' * 80}")
#     if en_rango:
#         muestra = random.sample(en_rango, min(5, len(en_rango)))
#         for pred, gt, sim in sorted(muestra, key=lambda x: -x[2]):
#             match = "✅ MISMO" if pred.lower() == gt.lower() else "⚠️  DISTINTO"
#             print(f"  sim={sim:.4f}  |  pred: {pred:<40} | gt: {gt:<40} | {match}")
#     else:
#         print("  (sin pares en este rango)")

# # Resumen cuantitativo
# print(f"\n{'=' * 80}")
# print("  RESUMEN: ¿Qué proporción de pares son realmente la misma película?")
# print("=" * 80)
# for label, low, high in rangos:
#     en_rango = [(p, g, s) for p, g, s in all_pairs if low <= s < high]
#     if en_rango:
#         exactos = sum(1 for p, g, _ in en_rango if p.strip().lower() == g.strip().lower())
#         print(f"  {label}: {exactos}/{len(en_rango)} exactos ({100*exactos/len(en_rango):.1f}%)")
#     else:
#         print(f"  {label}: sin pares")

# print(f"\nTotal de pares analizados: {len(all_pairs)}")
# print("\nConclusión: usar como argumento para justificar el threshold elegido.")
# print("Un threshold de 0.90 captura variaciones menores de título (año, prefijos)")
# print("sin introducir falsos positivos de películas diferentes.")